# 02. Portfolio Segmentation

Extracts the FR Y-9C line items required for the projection from M&T Bank
Corporation's 2025 Q4 filing, and verifies that segment-level items reconcile
to their reported totals.

Item codes are fixed here and reused for every quarter in the estimation
sample.

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

raw_y9c = Path("..") / "data" / "raw" / "y9c"
processed = Path("..") / "data" / "processed"

RSSD = 1037003
ANCHOR_FILE = raw_y9c / "FRY9C_1037003_20251231.csv"

y9c = pd.read_csv(ANCHOR_FILE)
y9c["Value"] = pd.to_numeric(y9c["Value"], errors="coerce")

print(f"{y9c.shape[0]} line items")

1647 line items


## Item definitions

Loan balances are point-in-time. Charge-offs, recoveries, and income statement
items are reported on a calendar year-to-date basis and require differencing to
recover quarterly values. That transformation is applied in notebook 03.

In [3]:
LOAN_SEGMENTS = {
    "BHDM5367": "RE: 1-4 family first lien",
    "BHDM5368": "RE: 1-4 family junior lien",
    "BHDM1797": "RE: Home equity revolving",
    "BHDM1460": "RE: Multifamily",
    "BHCKF158": "RE: 1-4 family construction",
    "BHCKF159": "RE: Other construction and land",
    "BHDM1420": "RE: Farmland",
    "BHCKF160": "RE: CRE owner-occupied",
    "BHCKF161": "RE: CRE other",
    "BHCK1763": "C&I: US addressees",
    "BHCK1764": "C&I: Non-US addressees",
    "BHCKB538": "Consumer: Credit card",
    "BHCKB539": "Consumer: Other revolving",
    "BHCKK137": "Consumer: Automobile",
    "BHCKK207": "Consumer: Other",
    "BHCK1590": "Agricultural",
    "BHDM2165": "Lease financing",
    "BHCK1292": "Depository institutions",
    "BHCKJ454": "Nondepository financial institutions",
    "BHCKJ451": "All other loans",
    "BHCK1545": "Other loans",
}

CHARGE_OFFS = {
    "BHCKC234": "RE: 1-4 family first lien",
    "BHCKC235": "RE: 1-4 family junior lien",
    "BHCK5411": "RE: Home equity revolving",
    "BHCK3588": "RE: Multifamily",
    "BHCKC891": "RE: 1-4 family construction",
    "BHCKC893": "RE: Other construction and land",
    "BHCK3584": "RE: Farmland",
    "BHCKC895": "RE: CRE owner-occupied",
    "BHCKC897": "RE: CRE other",
    "BHCK4652": "RE: Non-US addressees",
    "BHCK4645": "C&I: US addressees",
    "BHCK4646": "C&I: Non-US addressees",
    "BHCKB514": "Consumer: Credit card",
    "BHCKK129": "Consumer: Automobile",
    "BHCKK205": "Consumer: Other",
    "BHCK4655": "Agricultural",
    "BHCK4643": "Foreign governments",
    "BHCK4644": "All other loans",
    "BHCKC880": "All other leases",
    "BHCKF185": "Consumer leases",
}

RECOVERIES = {
    "BHCKC217": "RE: 1-4 family first lien",
    "BHCKC218": "RE: 1-4 family junior lien",
    "BHCK5412": "RE: Home equity revolving",
    "BHCK3589": "RE: Multifamily",
    "BHCKC892": "RE: 1-4 family construction",
    "BHCKC894": "RE: Other construction and land",
    "BHCK3585": "RE: Farmland",
    "BHCKC896": "RE: CRE owner-occupied",
    "BHCKC898": "RE: CRE other",
    "BHCK4662": "RE: Non-US addressees",
    "BHCK4617": "C&I: US addressees",
    "BHCK4618": "C&I: Non-US addressees",
    "BHCKB515": "Consumer: Credit card",
    "BHCKK133": "Consumer: Automobile",
    "BHCKK206": "Consumer: Other",
    "BHCK4665": "Agricultural",
    "BHCK4627": "Foreign governments",
    "BHCK4628": "All other loans",
    "BHCKF188": "All other leases",
    "BHCKF187": "Consumer leases",
}

CAPITAL = {
    "BHCAP859": "CET1 capital",
    "BHCAA223": "Total risk-weighted assets",
    "BHCAP793": "CET1 capital ratio",
    "BHCKS581": "Standardized market risk RWA",
}

INCOME = {
    "BHCK4074": "Net interest income",
    "BHCK4079": "Total noninterest income",
    "BHCK4093": "Total noninterest expense",
    "BHCK4302": "Applicable income taxes",
    "BHCK4301": "Pre-tax income",
    "BHCK4598": "Cash dividends: preferred",
    "BHCK4460": "Cash dividends: common",
}

CONTROL_TOTALS = {
    "BHCK2122": "Total loans and leases",
    "BHCK1410": "Loans secured by real estate",
    "BHDM1975": "Loans to individuals",
    "BHCK4635": "Total charge-offs",
    "BHCK4605": "Total recoveries",
    "BHCK2170": "Total assets",
}

## Reconciliation

Each set of segment-level items is checked against its reported total. A
non-zero difference indicates that an item has been misclassified, omitted, or
double counted.

In [5]:
def val(code):
    return y9c.loc[y9c["ItemName"] == code, "Value"].iloc[0]


def reconcile(name, codes, total_code):
    components = y9c.loc[y9c["ItemName"].isin(codes), "Value"].sum()
    total = val(total_code)
    diff = components - total
    status = "PASS" if diff == 0 else "FAIL"
    print(f"{name:24s} {components:>14,.0f} {total:>14,.0f} {diff:>10,.0f}  {status}")
    return diff


print(f"{'Check':24s} {'Components':>14s} {'Reported':>14s} {'Diff':>10s}")
print("-" * 68)

reconcile("Loan segments", LOAN_SEGMENTS, "BHCK2122")
reconcile("Charge-offs", CHARGE_OFFS, "BHCK4635")
reconcile("Recoveries", RECOVERIES, "BHCK4605")

Check                        Components       Reported       Diff
--------------------------------------------------------------------
Loan segments               138,809,737    138,809,737          0  PASS
Charge-offs                     746,153        746,153          0  PASS
Recoveries                      192,876        192,876          0  PASS


np.float64(0.0)

## Anchor balance sheet

Loan balances by segment as of 2025 Q4. These are the exposures to which
projected loss rates are applied.

In [6]:
def extract(codes, label="Item"):
    out = y9c.loc[y9c["ItemName"].isin(codes), ["ItemName", "Value"]].copy()
    out[label] = out["ItemName"].map(codes)
    return out[["ItemName", label, "Value"]].sort_values("Value", ascending=False)


loans = extract(LOAN_SEGMENTS, "Segment")
loans["Share of total"] = loans["Value"] / val("BHCK2122") * 100

print(loans.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

ItemName                              Segment         Value  Share of total
BHCK1763                   C&I: US addressees 32,904,960.00           23.71
BHDM5367            RE: 1-4 family first lien 24,793,482.00           17.86
BHCKK207                      Consumer: Other 14,798,677.00           10.66
BHCKF161                        RE: CRE other 12,861,910.00            9.27
BHCKJ454 Nondepository financial institutions 12,536,517.00            9.03
BHCKF160               RE: CRE owner-occupied 10,386,357.00            7.48
BHDM1460                      RE: Multifamily  6,840,992.00            4.93
BHCKK137                 Consumer: Automobile  5,167,727.00            3.72
BHDM1797            RE: Home equity revolving  4,862,040.00            3.50
BHCKJ451                      All other loans  3,732,700.00            2.69
BHCKF159      RE: Other construction and land  3,499,384.00            2.52
BHDM2165                      Lease financing  2,747,135.00            1.98
BHCK1545    

## Capital position

The 2025 Q4 capital ratio is the starting point for the projection. The
Federal Reserve states that the capital ratio at the end of 2025 served as the
starting point for the 2026 stress test.

Standardized market risk-weighted assets are zero, confirming that M&T is not
subject to the market risk capital rule and therefore not subject to the
global market shock component of the supervisory stress test.

In [7]:
capital = extract(CAPITAL, "Item")
print(capital.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

cet1 = val("BHCAP859")
rwa = val("BHCAA223")
print(f"\nCET1 / RWA = {cet1 / rwa * 100:.4f}%")
print(f"Reported   = {val('BHCAP793'):.4f}%")

ItemName                         Item            Value
BHCAA223   Total risk-weighted assets 161,891,600.0000
BHCAP859                 CET1 capital  17,551,365.0000
BHCAP793           CET1 capital ratio          10.8414
BHCKS581 Standardized market risk RWA           0.0000

CET1 / RWA = 10.8414%
Reported   = 10.8414%


## Income statement

Values are calendar year-to-date as of 2025 Q4 and therefore represent the
full year 2025. Quarterly values are recovered by differencing consecutive
filings within each calendar year; this is applied in notebook 03.

PPNR is defined here as net interest income plus noninterest income less
noninterest expense.

In [8]:
income = extract(INCOME, "Item")
print(income.to_string(index=False, float_format=lambda x: f"{x:,.0f}"))

nii = val("BHCK4074")
nonint_inc = val("BHCK4079")
nonint_exp = val("BHCK4093")
ppnr = nii + nonint_inc - nonint_exp

net_co = val("BHCK4635") - val("BHCK4605")
tax_rate = val("BHCK4302") / val("BHCK4301")

print(f"\nPPNR, full year 2025:        {ppnr:>14,.0f}")
print(f"Net charge-offs, 2025:       {net_co:>14,.0f}")
print(f"Effective tax rate:          {tax_rate:>14.1%}")

ItemName                      Item     Value
BHCK4074       Net interest income 6,948,080
BHCK4093 Total noninterest expense 5,366,094
BHCK4301            Pre-tax income 3,692,210
BHCK4079  Total noninterest income 2,613,465
BHCK4460    Cash dividends: common   899,637
BHCK4302   Applicable income taxes   841,227
BHCK4598 Cash dividends: preferred   146,283

PPNR, full year 2025:             4,195,451
Net charge-offs, 2025:              553,277
Effective tax rate:                   22.8%


## Outputs

In [9]:
processed.mkdir(parents=True, exist_ok=True)

item_map = {}
for group, mapping in [
    ("loan_balance", LOAN_SEGMENTS),
    ("charge_off", CHARGE_OFFS),
    ("recovery", RECOVERIES),
    ("capital", CAPITAL),
    ("income", INCOME),
    ("control_total", CONTROL_TOTALS),
]:
    for code, label in mapping.items():
        item_map[code] = {"group": group, "label": label}

item_map = pd.DataFrame.from_dict(item_map, orient="index")
item_map.index.name = "item_code"
item_map = item_map.reset_index()

out_path = processed / "y9c_item_map.csv"
item_map.to_csv(out_path, index=False)

print(f"Written: {out_path}")
print(f"{item_map.shape[0]} item codes across {item_map['group'].nunique()} groups")
item_map.groupby("group").size()

Written: ..\data\processed\y9c_item_map.csv
78 item codes across 6 groups


group
capital           4
charge_off       20
control_total     6
income            7
loan_balance     21
recovery         20
dtype: int64

## Summary

Item codes for M&T Bank Corporation (RSSD 1037003) fixed from the 2025 Q4
FR Y-9C filing.

| Check | Result |
|---|---|
| Loan segments sum to BHCK2122 | Pass |
| Charge-offs sum to BHCK4635 | Pass |
| Recoveries sum to BHCK4605 | Pass |
| CET1 / RWA equals reported ratio | Pass, 10.8414% |

**Anchor position, 2025 Q4.** Total loans of 138.8 billion, CET1 capital of
17.6 billion against risk-weighted assets of 161.9 billion.

**Portfolio concentration.** Commercial and industrial lending, first lien
residential mortgages, and commercial real estate together account for
approximately 58 percent of total loans. Commercial real estate exposure is
material relative to the severity of the 2026 scenario, in which commercial
real estate prices decline by approximately 39 percent.

**Not subject to the global market shock.** Standardized market
risk-weighted assets are zero.

**Output.** `data/processed/y9c_item_map.csv` maps each item code to its
group and label, and is consumed by notebook 03.

In [13]:
chi = pd.read_csv(
    raw_y9c / "bhcf1112.csv",
    skiprows=[1],
    low_memory=False,
)

print(chi.shape)
print(f"M&T present: {(chi['RSSD9001'] == 1037003).sum()} row(s)")

our_codes = set(item_map["item_code"])
available = our_codes & set(chi.columns)
missing = our_codes - set(chi.columns)

print(f"\nOur codes: {len(our_codes)}")
print(f"Available: {len(available)}")
print(f"Missing:   {len(missing)}")
print(sorted(missing))

(5147, 2268)
M&T present: 1 row(s)

Our codes: 78
Available: 74
Missing:   4
['BHCAA223', 'BHCAP793', 'BHCAP859', 'BHCKS581']


In [11]:
item_map = pd.read_csv(processed / "y9c_item_map.csv")

In [14]:
print("BHDM1975" in chi.columns)
print("BHCKB538" in chi.columns)
print("BHCKB514" in chi.columns)

True
True
True
